# Week 1: Gaia Stellar Data Exploration

**Goal:** Query real star data from the Gaia DR3 catalog, explore it raw, understand why cleaning matters, engineer features, and build the Hertzsprung-Russell (HR) diagram.

**What is Gaia?**  
A European Space Agency satellite that has measured the positions, distances, and brightness of ~1.8 billion stars. Data is queried live using ADQL — a dialect of SQL for astronomical archives.

**What is the HR Diagram?**  
A scatter plot of color (temperature proxy) vs. absolute magnitude (true brightness). Stars cluster into distinct sequences revealing their evolutionary stage:  
- **Main sequence** — stars fusing hydrogen (like our Sun)  
- **Red giant branch** — stars that have exhausted their core hydrogen  
- **White dwarfs** — dense stellar remnants, hot but dim  

---
**Flow:** `Install → Import → Query → Raw Explore → Clean → Feature Engineering → Visualize`

## Step 0: Install Dependencies

Run once if `astroquery` is not installed.

In [ ]:
%pip install astroquery

Defaulting to user installation because normal site-packages is not writeable
  Using cached astroquery-0.4.11-py3-none-any.whl.metadata (6.5 kB)
  Using cached astropy-7.2.0-cp311-abi3-macosx_11_0_arm64.whl.metadata (10 kB)
  Using cached html5lib-1.1-py2.py3-none-any.whl.metadata (16 kB)
  Using cached keyring-25.7.0-py3-none-any.whl.metadata (21 kB)
  Using cached pyvo-1.8.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached pyerfa-2.0.1.5-cp39-abi3-macosx_11_0_arm64.whl.metadata (5.7 kB)
  Using cached astropy_iers_data-0.2026.5.18.1.11.28-py3-none-any.whl.metadata (3.4 kB)
  Using cached webencodings-0.5.1-py2.py3-none-any.whl.metadata (2.1 kB)
  Using cached jaraco.classes-3.4.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached jaraco_functools-4.5.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached jaraco_context-6.1.2-py3-none-any.whl.metadata (4.2 kB)
  Using cached more_itertools-11.0.2-py3-none-any.whl.metadata (41 kB)
Using cached astroquery-0.4.11-py3-none-any.whl (11.1 MB)

## Imports

All imports in one place — run this before any other cell.

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

from src.data_fetch import fetch_gaia_sample
from src.data_clean import clean_sample, add_features
from src import visualize

## Step 1: Setup & Network Config

`astroquery` needs a higher timeout for the Gaia archive. The SSL override is a workaround for corporate network inspection that breaks the default certificate chain — it disables verification for this session.

> **Note:** SSL and timeout config is handled automatically inside `fetch_gaia_sample()` — no manual setup needed.

## Step 2: Query the Gaia Archive

We fetch 10,000 stars from `gaiadr3.gaia_source` using ADQL.

| Column | Meaning |
|--------|---------|
| `parallax` | Angular shift in milliarcseconds — used to compute distance |
| `parallax_error` | Measurement uncertainty on parallax |
| `phot_g_mean_mag` | Apparent brightness in the G (green) band |
| `bp_rp` | Color index (Blue − Red). Low = hot/blue, high = cool/red |

In [ ]:
df = fetch_gaia_sample(top_n=10000)
df.head()

## Step 3: Raw Exploration — Spot the Problems

Before cleaning, compute distance naively to see what goes wrong.  
`distance_pc = 1000 / parallax_mas` — but some parallax values are negative (noise artifact on very distant stars), giving **unphysical negative distances**:

In [ ]:
df["distance_pc"] = 1000 / df["parallax"]
df["distance_pc"]

0         2822.223410
1          318.555530
2         1588.183944
3         1823.475994
4         -718.199066
            ...      
9995      3156.756427
9996      3909.559294
9997    352001.769850
9998       349.580154
9999      3644.652835
Name: distance_pc, Length: 10000, dtype: float64

### Parallax Distribution (raw)

Notice the negative tail on the left — these are the measurement noise artifacts we need to remove:

### Labeled Parallax Histogram

Same plot with proper axis labels — most stars cluster at small parallax (distant), with a tail of nearby bright stars at high values:

In [ ]:
visualize.plot_parallax_hist(df)

### Log-Scale Parallax Histogram

Same data on a **log y-axis** — makes rare high-parallax stars (nearby) visible alongside the dense bulk of distant stars. Without log scale, the tall distant-star bar crushes everything else.

In [ ]:
visualize.plot_parallax_hist_log(df)

## Step 4: Clean the Data

Three quality filters — each targets a specific class of bad measurement:

| Filter | Removes | Why |
|--------|---------|-----|
| `parallax > 0` | Negative parallax | Unphysical — gives negative distance |
| `parallax / parallax_error > 5` | Low signal-to-noise stars | Distance uncertainty > 25% → absolute magnitude unreliable |
| `bp_rp` not null | Stars with no color | Can't be placed on the HR diagram |

In [ ]:
clean = clean_sample(df)

## Step 5: Feature Engineering

Two derived columns needed for the HR diagram — neither exists in the raw Gaia data:

**Distance in parsecs:**  
`distance_pc = 1000 / parallax_mas`  
1 parsec ≈ 3.26 light-years. Parallax in milliarcseconds inverts directly.

**Absolute magnitude:**  
`M = m − 5 × log₁₀(d / 10)`  
The **distance modulus** — strips out distance so we compare true intrinsic stellar brightness. At d = 10 pc, M = m (the standard reference point).

In [ ]:
clean = add_features(clean)
print(clean[["bp_rp", "absolute_mag", "distance_pc"]].describe())

## Step 6: Visualize

### 6a. HR Diagram — Density Map

Hex-bin plot with log color scale reveals structure across orders of magnitude.  
**Y-axis inverted** — astronomy convention: bright (low magnitude) at top.

Look for:
- **Diagonal band** top-left → bottom-right → **main sequence**
- **Clump upper-right** (bright + red) → **red giants**
- **Sparse lower-left** (dim + blue/white) → **white dwarfs**

In [ ]:
visualize.plot_hr_density(clean)

### 6a-ii. HR Diagram — Scatter Plot

Same data as the density map above but as a plain scatter plot — each dot is one star.  
Useful for spotting individual outliers and the overall population shape before density encoding hides detail.  
`s=1, alpha=0.5` keeps it readable at 10k points.

In [ ]:
visualize.plot_hr_scatter(clean)

### 6b. Absolute Magnitude Distribution

Peak around M ≈ 4–6, dominated by main-sequence stars. Our Sun sits at M ≈ 4.83 — right in the middle.

In [ ]:
visualize.plot_abs_mag_hist(clean)

### 6c. Distance vs. Brightness — Malmquist Bias

At large distances only intrinsically bright stars are detectable above Gaia's flux limit. The **missing lower-right corner** (faint + far = invisible) makes this selection bias visible.

In [ ]:
visualize.plot_distance_vs_mag(clean)

---

## Summary

| Step | What we did | Key insight |
|------|-------------|-------------|
| Query | 10k stars from Gaia DR3 via ADQL | Real observational data |
| Raw explore | Naive distance → negative values | Shows exactly why cleaning is necessary |
| Clean | Removed negative parallax, low-SNR, missing color | Each filter has a physical motivation |
| Features | `distance_pc` and `absolute_mag` | Distance modulus corrects for star distance |
| Visualize | HR diagram, magnitude hist, Malmquist plot | Sample biases are visible in the data |

**Next:** [`week2_kmeans_clustering.ipynb`](week2_kmeans_clustering.ipynb) — K-Means to recover stellar populations without labels.